# Configuración Inicial y Carga de Datos

En esta sección importamos las librerías necesarias para la manipulación de datos y, específicamente, las herramientas de scikit-learn que utilizaremos para el preprocesamiento (codificación y escalamiento).

In [16]:
# IMPORTACIÓN DE LIBRERÍAS
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Librerías para Preprocesamiento (Scikit-Learn)
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer

sns.set(style="whitegrid")
pd.set_option('display.max_columns', None)

Cargamos el dataset que ya limpiamos en la fase anterior (sin 'unknowns' en job/education)

In [17]:
df = pd.read_csv('bank_depurado.csv')
print(f"Dimensiones: {df.shape}")
df.head()

Dimensiones: (11162, 18)


,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,deposit,tipo_cliente_pdays
0,59,admin.,married,secondary,no,2343,yes,no,unknown,5,may,1042,1,-1,0,unknown,yes,Cliente Nuevo
1,56,admin.,married,secondary,no,45,no,no,unknown,5,may,1467,1,-1,0,unknown,yes,Cliente Nuevo
2,41,technician,married,secondary,no,1270,yes,no,unknown,5,may,1389,1,-1,0,unknown,yes,Cliente Nuevo
3,55,services,married,secondary,no,2476,yes,no,unknown,5,may,579,1,-1,0,unknown,yes,Cliente Nuevo
4,54,admin.,married,tertiary,no,184,no,no,unknown,5,may,673,2,-1,0,unknown,yes,Cliente Nuevo


Antes de aplicar cualquier transformación, realizamos una inspección rápida para confirmar los tipos de datos de cada variable y asegurarnos de que no hay valores nulos pendientes.

In [18]:
# Revisamos los tipos de datos para planear la estrategia
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 11162 entries, 0 to 11161
Data columns (total 18 columns):
 #   Column              Non-Null Count  Dtype 
---  ------              --------------  ----- 
 0   age                 11162 non-null  int64 
 1   job                 11162 non-null  object
 2   marital             11162 non-null  object
 3   education           11162 non-null  object
 4   default             11162 non-null  object
 5   balance             11162 non-null  int64 
 6   housing             11162 non-null  object
 7   loan                11162 non-null  object
 8   contact             11162 non-null  object
 9   day                 11162 non-null  int64 
 10  month               11162 non-null  object
 11  duration            11162 non-null  int64 
 12  campaign            11162 non-null  int64 
 13  pdays               11162 non-null  int64 
 14  previous            11162 non-null  int64 
 15  poutcome            11162 non-null  object
 16  deposit             11

# Clasificación de Variables


Para aplicar las técnicas de preprocesamiento adecuadas, separamos las variables en dos grupos:
* **Numéricas:** Se les aplicará escalamiento.
* **Categóricas:** Se les aplicará codificación (encoding).
La variable objetivo (`deposit`) se separa para tratarla individualmente al final.

In [19]:
# Definimos la variable objetivo
target = 'deposit'

# Identificamos las columnas numéricas
cols_numericas = df.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Identificamos las columnas categóricas excluyendo la objetivo
cols_categoricas = df.select_dtypes(include=['object']).columns.tolist()
if target in cols_categoricas:
    cols_categoricas.remove(target)

print(f"Variables Numéricas ({len(cols_numericas)}): {cols_numericas}")
print(f"Variables Categóricas ({len(cols_categoricas)}): {cols_categoricas}")
print(f"Variable Objetivo: {target}")

Variables Numéricas (7): ['age', 'balance', 'day', 'duration', 'campaign', 'pdays', 'previous']
Variables Categóricas (10): ['job', 'marital', 'education', 'default', 'housing', 'loan', 'contact', 'month', 'poutcome', 'tipo_cliente_pdays']
Variable Objetivo: deposit


Esta separación nos permite automatizar el proceso de transformación en los siguientes pasos, asegurando que aplicamos MinMaxScaler solo a los números y OneHot/LabelEncoder solo al texto.

# Codificación de Variables Categóricas (Encoding)


Los algoritmos de aprendizaje automático requieren entradas numéricas.
* Para la variable objetivo (`deposit`), utilizamos **Label Encoding** (binario).
* Para las variables predictoras categóricas, utilizamos **One-Hot Encoding** (variables dummy), ya que no existe un orden ordinal intrínseco (por ejemplo, 'casado' no es mayor que 'soltero').

In [20]:
# Codificamos la variable objetivo 'deposit' (yes=1, no=0)
le = LabelEncoder()
df[target] = le.fit_transform(df[target])

print(f"Variable objetivo '{target}' codificada: {df[target].unique()}")

# Aplicamos One-Hot Encoding a las variables predictoras categóricas
df_codificado = pd.get_dummies(df, columns=cols_categoricas, drop_first=True)

print("\nDimensiones después del One-Hot Encoding:")
print(df_codificado.shape)

df_codificado.head(10)

Variable objetivo 'deposit' codificada: [1 0]

Dimensiones después del One-Hot Encoding:
(11162, 42)


,age,balance,day,duration,campaign,pdays,previous,deposit,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_married,marital_single,education_secondary,education_tertiary,default_yes,housing_yes,loan_yes,contact_telephone,contact_unknown,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,tipo_cliente_pdays_Cliente Recurrente
0,59,2343,5,1042,1,-1,0,1,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
1,56,45,5,1467,1,-1,0,1,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
2,41,1270,5,1389,1,-1,0,1,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
3,55,2476,5,579,1,-1,0,1,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
4,54,184,5,673,2,-1,0,1,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
5,42,0,5,562,2,-1,0,1,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,True,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
6,56,830,6,1201,1,-1,0,1,False,False,False,True,False,False,False,False,False,False,True,False,False,True,False,True,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
7,60,545,6,1030,1,-1,0,1,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
8,37,1,6,608,1,-1,0,1,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
9,28,5090,6,1297,3,-1,0,1,False,False,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False


Observamos que el número de columnas aumentó considerablemente. Esto es normal, ya que variables como job o month se han desplegado en múltiples columnas binarias (0 o 1) para representar cada categoría sin imponer un orden falso.

# Escalamiento de Variables Numéricas


Las variables numéricas tienen rangos muy distintos (por ejemplo, `age` vs `balance`). Para evitar que las variables con mayor magnitud dominen el modelo, aplicamos **MinMaxScaling**.

Esto transforma todos los valores numéricos para que estén dentro del rango [0, 1], preservando la distribución original pero normalizando la escala.

In [21]:
scaler = MinMaxScaler()

# Aplicamos el escalamiento  a las columnas numéricas)
df_codificado[cols_numericas] = scaler.fit_transform(df_codificado[cols_numericas])

# Verificamos el resultado revisando los máximos y mínimos
print("Rango de las variables numéricas después del escalamiento:")
print(df_codificado[cols_numericas].describe().loc[['min', 'max']])

# Vista previa final del dataset
df_codificado.head(10)

Rango de las variables numéricas después del escalamiento:
     age  balance  day  duration  campaign  pdays  previous
min  0.0      0.0  0.0       0.0       0.0    0.0       0.0
max  1.0      1.0  1.0       1.0       1.0    1.0       1.0


,age,balance,day,duration,campaign,pdays,previous,deposit,job_blue-collar,job_entrepreneur,job_housemaid,job_management,job_retired,job_self-employed,job_services,job_student,job_technician,job_unemployed,marital_married,marital_single,education_secondary,education_tertiary,default_yes,housing_yes,loan_yes,contact_telephone,contact_unknown,month_aug,month_dec,month_feb,month_jan,month_jul,month_jun,month_mar,month_may,month_nov,month_oct,month_sep,poutcome_other,poutcome_success,poutcome_unknown,tipo_cliente_pdays_Cliente Recurrente
0,0.532468,0.104371,0.133333,0.268110,0.000000,0.0,0.0,1,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
1,0.493506,0.078273,0.133333,0.377675,0.000000,0.0,0.0,1,False,False,False,False,False,False,False,False,False,False,True,False,True,False,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
2,0.298701,0.092185,0.133333,0.357566,0.000000,0.0,0.0,1,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
3,0.480519,0.105882,0.133333,0.148750,0.000000,0.0,0.0,1,False,False,False,False,False,False,True,False,False,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
4,0.467532,0.079851,0.133333,0.172983,0.016129,0.0,0.0,1,False,False,False,False,False,False,False,False,False,False,True,False,False,True,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
5,0.311688,0.077762,0.133333,0.144367,0.016129,0.0,0.0,1,False,False,False,True,False,False,False,False,False,False,False,True,False,True,False,True,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
6,0.493506,0.087188,0.166667,0.309100,0.000000,0.0,0.0,1,False,False,False,True,False,False,False,False,False,False,True,False,False,True,False,True,True,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
7,0.545455,0.083951,0.166667,0.265017,0.000000,0.0,0.0,1,False,False,False,False,True,False,False,False,False,False,False,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
8,0.246753,0.077773,0.166667,0.156226,0.000000,0.0,0.0,1,False,False,False,False,False,False,False,False,True,False,True,False,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False
9,0.129870,0.135569,0.166667,0.333849,0.032258,0.0,0.0,1,False,False,False,False,False,False,True,False,False,False,False,True,True,False,False,True,False,False,True,False,False,False,False,False,False,False,True,False,False,False,False,False,True,False


Como se observa en la tabla de descripción, todas las variables numéricas ahora tienen un valor mínimo de 0.0 y un máximo de 1.0. El dataset está completamente homogeneizado.

# Exportación de Datos Preprocesados



Finalmente, se guarda el dataset transformado en un nuevo archivo CSV.

In [22]:
try:
    df_codificado.to_csv('data/bank_preprocesado.csv', index=False)
    print("rchivo 'bank_preprocesado.csv' guardado exitosamente en la carpeta /data.")
except OSError:
    df_codificado.to_csv('bank_preprocesado.csv', index=False)
    print("rchivo 'bank_preprocesado.csv' guardado exitosamente en la raíz.")

print(f"Dimensiones finales del dataset exportado: {df_codificado.shape}")

rchivo 'bank_preprocesado.csv' guardado exitosamente en la raíz.
Dimensiones finales del dataset exportado: (11162, 42)


Este archivo contiene todas las variables numéricas escaladas y las categóricas codificadas, listo para ser utilizado en etapas posteriores de modelado predictivo.